In [1]:
# gensim: 专门训练词向量的库
# konlpy: 专门处理韩语的库（因为我们要用 Naver 影评数据）
# tqdm: 进度条工具，让你看着不焦虑
!pip install gensim konlpy tqdm urllib3


   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   --- ------------------------------------ 2.4/24.4 MB 11.6 MB/s eta 0:00:02
   ----- ---------------------------------- 3.4/24.4 MB 8.0 MB/s eta 0:00:03
   -------- ------------------------------- 5.2/24.4 MB 9.0 MB/s eta 0:00:03
   ----------- ---------------------------- 7.1/24.4 MB 8.3 MB/s eta 0:00:03
   --------------- ------------------------ 9.4/24.4 MB 9.0 MB/s eta 0:00:02
   ------------------ --------------------- 11.5/24.4 MB 9.1 MB/s eta 0:00:02
   -------------------- ------------------- 12.6/24.4 MB 8.5 MB/s eta 0:00:02
   --------------------- ------------------ 13.4/24.4 MB 7.9 MB/s eta 0:00:02
   ------------------------ --------------- 14.7/24.4 MB 7.7 MB/s eta 0:00:02
   ------------------------- -------------- 15.7/24.4 MB 7.4 MB/s eta 0:00:02
   --------------------------- ------------ 16.8/24.4 MB 7.5 MB/s eta 0:00:02
   ------------------------------ --------- 18.9/24.4 MB 7.5 MB/s eta 0:00:0

In [2]:
import urllib.request
import pandas as pd

#import data
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="ratings_train.txt")

#read data
train_data = pd.read_table('ratings_train.txt')

#Clean data
train_data = train_data.dropna(how='any')

print(f"finish downloading and we get {len(train_data)} movies comments")
print(train_data.head())#1 positive 0 negative



finish downloading and we get 149995 movies comments
         id                                           document  label
0   9976970                                아 더빙.. 진짜 짜증나네요 목소리      0
1   3819312                  흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나      1
2  10265843                                  너무재밓었다그래서보는것을추천한다      0
3   9045019                      교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정      0
4   6483659  사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...      1


In [4]:
from konlpy.tag import Okt
from tqdm import tqdm

#Initialize the Korean tokenizer
okt = Okt()

sample_data = train_data['document'][:len(train_data)]

sentences = []
print("The Korean comments are currently undergoing word segmentation processing...")

for sentence in tqdm(sample_data):
    # Get Noun、Verb、Adjective
    # stem=True  restore words to their most original state
    temp_tokens = okt.pos(sentence, stem=True)
    clean_tokens = [word for word, tag in temp_tokens if tag in ['Noun', 'Verb', 'Adjective']]

    if clean_tokens:
        sentences.append(clean_tokens)

print(f"\n Word segmentation ends, you have the {len(sentences)} vocabulary bag ready for training.")
print("Eg.", sentences[0])

The Korean comments are currently undergoing word segmentation processing...


100%|██████████| 149995/149995 [01:55<00:00, 1302.19it/s]


 Word segmentation ends, you have the 148147 vocabulary bag ready for training.
Eg. ['더빙', '진짜', '짜증나다', '목소리']


In [5]:
from gensim.models import Word2Vec, FastText

# 1 Train CBOW model (sg=0)
model_cbow = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, workers=4, sg=0)

# 2 Train Skip-gram model (sg=1)
model_sg = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, workers=4, sg=1)

# 3 Train FastText model
model_ft = FastText(sentences=sentences, vector_size=100, window=5, min_count=5, workers=4)

# 4 Save trained models
model_cbow.save("cbow.model")
model_sg.save("skipgram.model")
model_ft.save("fasttext.model")

print("Training completed and models saved.")

Training completed and models saved.


In [6]:
import torch
import torch.nn as nn

# Convert Gensim model to PyTorch Embedding layer
def gensim_to_pytorch(gensim_model):
    # Extract weight matrix from the trained model
    weights = torch.FloatTensor(gensim_model.wv.vectors)

    # Initialize PyTorch Embedding layer with pre-trained weights
    # freeze=True means weights won't change during further training
    return nn.Embedding.from_pretrained(weights, freeze=True)

# Load Skip-gram model into PyTorch
pytorch_embedding = gensim_to_pytorch(model_sg)
print(f"PyTorch Embedding Layer created: {pytorch_embedding}")

PyTorch Embedding Layer created: Embedding(13451, 100)


In [7]:
# Compare top 5 similar words for '영화' (Movie)
target_word = '영화'

print(f"Results for '{target_word}':")
print("-" * 30)
print("CBOW:", [w for w, s in model_cbow.wv.most_similar(target_word, topn=5)])
print("Skip-gram:", [w for w, s in model_sg.wv.most_similar(target_word, topn=5)])
print("FastText:", [w for w, s in model_ft.wv.most_similar(target_word, topn=5)])

Results for '영화':
------------------------------
CBOW: ['영화로', '독립영화', '작품', '애니메이션', '다큐']
Skip-gram: ['멜로영화', '애니메이션영화', '괴수영화', '인도영화', '이처럼']
FastText: ['영화광', '극영화', '신영화', '민영화', '영화상']


*Conclusion & Analysis:
1.CBOW: Provides stable results for general categories('영화로', '독립영화', '작품', '애니메이션', '다큐').
2.Skip-gram: Better at capturing specific sub-genres('멜로영화', '애니메이션영화', '괴수영화', '인도영화', '이처럼').
3.FastText: Strongest at capturing morphological similarities (words containing '영화'), but can be sensitive to non-semantic character matches('영화광', '극영화', '신영화', '민영화', '영화상').
4.PyTorch Integration: Successfully initialized a pre-trained embedding layer with a vocabulary size of 13,451 and vector dimension of 100.